In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import math
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")


In [8]:
# ==============================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ==============================================================================
import pandas as pd
import numpy as np
import math
from scipy.sparse import csr_matrix
from scipy.special import expit
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# 2. DEFINICIÓN DE FUNCIONES AUXILIARES
# ==============================================================================
def filter_cold_start_fast(df, min_user=10, min_item=5):
    """
    Filtrado iterativo vectorizado para eliminar el problema de Cold Start.
    """
    active_mask = np.ones(len(df), dtype=bool)
    users_orig  = df['user_id'].values
    items_orig  = df['anime_id'].values

    prev_active = -1
    iteration   = 0

    while active_mask.sum() != prev_active:
        prev_active = active_mask.sum()
        iteration  += 1

        active_users = users_orig[active_mask]
        active_items = items_orig[active_mask]

        u_vals, u_counts = np.unique(active_users, return_counts=True)
        i_vals, i_counts = np.unique(active_items, return_counts=True)

        valid_u = set(u_vals[u_counts >= min_user])
        valid_i = set(i_vals[i_counts >= min_item])

        active_mask = (
            np.isin(users_orig, list(valid_u)) &
            np.isin(items_orig, list(valid_i))
        )
        print(f'  Iter {iteration}: {active_mask.sum():,} ratings | '
              f'{len(valid_u):,} usuarios | {len(valid_i):,} animes')

    print(f'\n✅ Convergencia en {iteration} iteraciones')
    return df[active_mask].reset_index(drop=True)

def build_sparse(df, n_users, n_items, val_col='rating'):
    """
    Construye una matriz dispersa CSR a partir de un DataFrame.
    """
    return csr_matrix(
        (df[val_col].values.astype('float32'),
         (df['user_id'].values.astype('int32'),
          df['anime_id'].values.astype('int32'))),
        shape=(n_users, n_items)
    )

# ==============================================================================
# 3. PIPELINE DE DATOS: CARGA, MUESTREO Y SPLIT
# ==============================================================================
print("Cargando el dataset original...")
df_rating_original = pd.read_csv("rating.csv") 

# --- NUEVO: MUESTREO DEL 10% DE USUARIOS ---
print("\nMuestreando un 10% de los usuarios para agilizar la ejecución...")
np.random.seed(42) # Semilla para que el resultado sea reproducible
todos_los_usuarios = df_rating_original['user_id'].unique()
usuarios_muestra = np.random.choice(todos_los_usuarios, size=int(len(todos_los_usuarios) * 0.10), replace=False)

# Filtramos el dataset para quedarnos solo con esos usuarios
df_rating_original = df_rating_original[df_rating_original['user_id'].isin(usuarios_muestra)].copy()
print(f"Total de ratings tras el muestreo: {len(df_rating_original):,}")
# -------------------------------------------

print("\nFiltrando ratings implícitos (-1)...")
df_rating = df_rating_original[df_rating_original['rating'] != -1].copy()

# OJO: Umbrales rebajados porque ahora tenemos una décima parte de la información
print("\nAplicando filtrado Cold Start adaptado (min_user=10, min_item=5)...")
df_rating = filter_cold_start_fast(df_rating, min_user=10, min_item=5)

print("\nRe-indexando usuarios e ítems de forma contigua...")
unique_users = df_rating['user_id'].unique()
unique_items = df_rating['anime_id'].unique()

user2idx = {id_: idx for idx, id_ in enumerate(unique_users)}
anime2idx = {id_: idx for idx, id_ in enumerate(unique_items)}

df_rating['user_id'] = df_rating['user_id'].map(user2idx)
df_rating['anime_id'] = df_rating['anime_id'].map(anime2idx)

NUM_USERS = len(user2idx)
NUM_ITEMS = len(anime2idx)

print(f"\nSeparando datos en Train (80%) y Test (20%)...")
df_train, df_test = train_test_split(
    df_rating, test_size=0.2, random_state=42,
    stratify=df_rating['user_id']
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print("Creando matrices dispersas (CSR)...")
R_train = build_sparse(df_train, NUM_USERS, NUM_ITEMS)
R_test  = build_sparse(df_test,  NUM_USERS, NUM_ITEMS)

print(f"\n¡Todo listo para correr de forma fluida! 🚀")
print(f"➡️ Usuarios únicos: {NUM_USERS:,}")
print(f"➡️ Animes únicos:   {NUM_ITEMS:,}")
print(f"➡️ R_train shape:   {R_train.shape} (Votos: {R_train.nnz:,})")
print(f"➡️ R_test shape:    {R_test.shape} (Votos: {R_test.nnz:,})")

Cargando el dataset original...

Muestreando un 10% de los usuarios para agilizar la ejecución...
Total de ratings tras el muestreo: 794,473

Filtrando ratings implícitos (-1)...

Aplicando filtrado Cold Start adaptado (min_user=10, min_item=5)...
  Iter 1: 630,324 ratings | 5,536 usuarios | 5,447 animes
  Iter 2: 630,297 ratings | 5,533 usuarios | 5,446 animes
  Iter 3: 630,297 ratings | 5,533 usuarios | 5,446 animes

✅ Convergencia en 3 iteraciones

Re-indexando usuarios e ítems de forma contigua...

Separando datos en Train (80%) y Test (20%)...
Creando matrices dispersas (CSR)...

¡Todo listo para correr de forma fluida! 🚀
➡️ Usuarios únicos: 5,533
➡️ Animes únicos:   5,446
➡️ R_train shape:   (5533, 5446) (Votos: 504,237)
➡️ R_test shape:    (5533, 5446) (Votos: 126,060)


In [9]:
from scipy.special import expit # Implementación rápida de la función logística (Sigmoide)

# BMF entrena una representación latente para cada nota posible
SCORES = np.arange(1, 11) # Notas del 1 al 10
NUM_SCORES = len(SCORES)

# Extraemos las coordenadas de la matriz de entrenamiento
coo_train = R_train.tocoo()
train_users = coo_train.row.astype(int)
train_items = coo_train.col.astype(int)
train_ratings = coo_train.data.astype('float32')
n_ratings = len(train_ratings)

In [10]:
# Hiperparámetros (puedes ajustarlos en la fase de optimización)
NUM_FACTORS_BMF = 20
LEARNING_RATE_BMF = 0.005
REGULARIZATION_BMF = 0.05
NUM_ITERATIONS_BMF = 10

# Inicialización de factores tridimensionales: (Scores, Usuarios/Items, Factores)
np.random.seed(42)
U_bmf = np.random.uniform(0, 1, (NUM_SCORES, NUM_USERS, NUM_FACTORS_BMF)).astype('float32')
V_bmf = np.random.uniform(0, 1, (NUM_SCORES, NUM_ITEMS, NUM_FACTORS_BMF)).astype('float32')

print("Comenzando el entrenamiento de BMF...")

for s_idx, score in enumerate(SCORES):
    print(f"Entrenando factores para el Score: {score}")
    
    # Binarización temporal (Target = 1 si la nota es igual al score actual, si no 0)
    target = (train_ratings == score).astype(float)
    
    for epoch in range(NUM_ITERATIONS_BMF):
        # Barajar en cada época (Stochastic Gradient Descent)
        idx = np.random.permutation(n_ratings)
        u_shuf = train_users[idx]
        i_shuf = train_items[idx]
        t_shuf = target[idx]
        
        for n in range(n_ratings):
            u = u_shuf[n]
            i = i_shuf[n]
            t = t_shuf[n]
            
            # Producto escalar y sigmoide (probabilidad)
            dot = np.dot(U_bmf[s_idx, u], V_bmf[s_idx, i])
            pred = expit(dot)
            error = t - pred # Derivada simplificada
            
            u_old = U_bmf[s_idx, u].copy()
            
            # Actualización de pesos
            U_bmf[s_idx, u] += LEARNING_RATE_BMF * (error * V_bmf[s_idx, i] - REGULARIZATION_BMF * U_bmf[s_idx, u])
            V_bmf[s_idx, i] += LEARNING_RATE_BMF * (error * u_old - REGULARIZATION_BMF * V_bmf[s_idx, i])
            
print("✅ Entrenamiento BMF finalizado.")

Comenzando el entrenamiento de BMF...
Entrenando factores para el Score: 1
Entrenando factores para el Score: 2
Entrenando factores para el Score: 3
Entrenando factores para el Score: 4
Entrenando factores para el Score: 5
Entrenando factores para el Score: 6
Entrenando factores para el Score: 7
Entrenando factores para el Score: 8
Entrenando factores para el Score: 9
Entrenando factores para el Score: 10
✅ Entrenamiento BMF finalizado.


In [11]:
coo_test = R_test.tocoo()
test_users = coo_test.row.astype(int)
test_items = coo_test.col.astype(int)
test_ratings = coo_test.data.astype('float32')

# Matrices para guardar la predicción final y su probabilidad máxima
best_preds = np.zeros(len(test_users))
best_probs = np.zeros(len(test_users)) - 1

for s_idx, score in enumerate(SCORES):
    # Evaluamos todos los pares de test de manera vectorizada
    dot = np.sum(U_bmf[s_idx, test_users] * V_bmf[s_idx, test_items], axis=1)
    prob = expit(dot)
    
    # Quedarnos con el score de mayor probabilidad
    mask = prob > best_probs
    best_probs[mask] = prob[mask]
    best_preds[mask] = score

# Cálculo de RMSE y MAE
from sklearn.metrics import mean_squared_error, mean_absolute_error

bmf_rmse = np.sqrt(mean_squared_error(test_ratings, best_preds))
bmf_mae = mean_absolute_error(test_ratings, best_preds)

print(f"BMF - RMSE: {bmf_rmse:.4f}")
print(f"BMF - MAE : {bmf_mae:.4f}")

BMF - RMSE: 2.0560
BMF - MAE : 1.4418


In [12]:
import math

N_RECOMMENDATIONS = 5
THETA = 7 # Consideramos un anime como "Relevante" si la nota es >= 7

def evaluate_ranking_metrics_bmf(test_users, test_items, test_ratings, best_preds):
    user_test_data = {}
    
    # Agrupar datos por usuario
    for u, i, true_r, pred_r in zip(test_users, test_items, test_ratings, best_preds):
        if u not in user_test_data:
            user_test_data[u] = {'items': [], 'true_ratings': [], 'pred_ratings': []}
        user_test_data[u]['items'].append(i)
        user_test_data[u]['true_ratings'].append(true_r)
        user_test_data[u]['pred_ratings'].append(pred_r)
        
    precisions, recalls, ndcgs = [], [], []
    
    for u, data in user_test_data.items():
        true_ratings = np.array(data['true_ratings'])
        pred_ratings = np.array(data['pred_ratings'])
        items = np.array(data['items'])
        
        # Identificamos los animes relevantes para este usuario (nota >= THETA)
        relevant_items = set(items[true_ratings >= THETA])
        
        if len(relevant_items) == 0:
            continue
            
        # Top-N animes basados en nuestra predicción de BMF
        top_n_idx = np.argsort(pred_ratings)[::-1][:N_RECOMMENDATIONS]
        top_n_items = items[top_n_idx]
        
        # Intersection entre el Top N y los relevantes
        hits = sum(1 for item in top_n_items if item in relevant_items)
        
        precisions.append(hits / N_RECOMMENDATIONS)
        recalls.append(hits / len(relevant_items))
        
        # Cálculo de nDCG
        dcg = 0
        for pos, idx in enumerate(top_n_idx):
            if items[idx] in relevant_items:
                dcg += (2**true_ratings[idx] - 1) / math.log2(pos + 2)
                
        # Ideal DCG (ordenando por la nota real)
        ideal_idx = np.argsort(true_ratings)[::-1][:N_RECOMMENDATIONS]
        idcg = sum((2**true_ratings[idx] - 1) / math.log2(pos + 2) for pos, idx in enumerate(ideal_idx) if true_ratings[idx] >= THETA)
        
        ndcgs.append(dcg / idcg if idcg > 0 else 0)
        
    avg_precision = np.mean(precisions)
    avg_recall = np.mean(recalls)
    avg_f1 = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) > 0 else 0
    avg_ndcg = np.mean(ndcgs)
    
    return avg_precision, avg_recall, avg_f1, avg_ndcg

precision, recall, f1, ndcg = evaluate_ranking_metrics_bmf(test_users, test_items, test_ratings, best_preds)

print(f"Precision@{N_RECOMMENDATIONS}: {precision:.4f}")
print(f"Recall@{N_RECOMMENDATIONS}: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"nDCG@{N_RECOMMENDATIONS}: {ndcg:.4f}")

Precision@5: 0.8287
Recall@5: 0.4888
F1 Score: 0.6149
nDCG@5: 0.6609
